# LatComm: запуск из своего форка

Этот ноутбук клонирует **[SalavatFayzullin/LatComm](https://github.com/SalavatFayzullin/LatComm)**,
устанавливает среду и запускает `kaggle/run.py` из клонированного репозитория.
Перед первым запуском **закоммитьте и отправьте папку `kaggle/` в свой форк**.

В Kaggle включите **Internet** и **GPU T4 ×2** или **P100**.
Данные авторов загружаются автоматически с Hugging Face.
По умолчанию выбран уменьшенный эксперимент **`PROFILE = "kaggle"`**:

- Qwen3.5-0.8B → Qwen3.5-0.8B; обе LLM заморожены.
- Stage 1: 2048 текстов, одна эпоха, 64 обновления.
- Stage 2: по 128 вопросов из каждого из 12 наборов, три эпохи, 144 обновления.
- Global batch 32, microbatch 1; на двух GPU накопление составляет 16 микробатчей.
- Тест: по 16 вопросов из каждого набора, всего 192.

Архитектура, два этапа и функция потерь соответствуют методу статьи. Объём данных,
batch и warmup уменьшены; метрики такого эксперимента не воспроизводят таблицы статьи.
Исходный архив Stage 1 (около 2,8 GB) всё равно скачивается и просматривается потоком.
`PROFILE = "full"` явно включает исходный полный бюджет, `"smoke"` — короткую проверку цепочки.

Для ночного эксперимента выберите **`PROFILE = "overnight"`**. Он увеличивает Stage 1 до
45 056 текстов (1 408 обновлений), сохраняя объём Stage 2 и теста. По темпу прошлого запуска
на T4 ×2 это около 15 часов вычислений. В одной GPU-сессии Kaggle доступно до 12 часов.

Первую часть запустите через **Save Version → Save & Run All**, оставив `RESTORE = None`.
Ноутбук запустит только Stage 1 и сохранит чекпоинт примерно через 9,5 часа обучения.
Затем подключите Output сохранённой версии как Input в чистой сессии и укажите `RESTORE`
на каталог `latcomm-overnight` из этого Input. С тем же профилем и T4 ×2 снова выберите
**Save & Run All**. Ноутбук продолжит обучение и после финального теста сравнит правильные
и подменённые пакеты. Подготовленные данные сохраняются рядом с весами и метриками.

Новый запуск берёт `REPO_REF`. Для продолжения обучения используется SHA из сохранённого
`manifest.json`, чтобы обновление ветки main не изменило код посреди эксперимента.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/SalavatFayzullin/LatComm.git"
REPO_REF = "main"                       # branch/tag for a new experiment
SOURCE_COMMIT = None                    # optional full SHA for a fixed version
REPO_DIR = "/kaggle/working/LatComm"     # source code from your fork
PROFILE = "kaggle"                     # choose "overnight" for ~15h across Kaggle sessions
WORKDIR = f"/kaggle/working/latcomm-{PROFILE}"  # checkpoints/results, separate from source
PAIR = None                           # kaggle/overnight: 0.8B -> 0.8B; full/smoke: 4B -> 0.8B
RESTORE = None                        # set to the saved WORKDIR under /kaggle/input/... to continue
ACTION = "stage1" if PROFILE == "overnight" and RESTORE is None else "all"
NUM_GPUS = 2 if PROFILE == "overnight" else 0  # 0 = all visible GPUs
SESSION_STEPS = 0                      # 0: finish training; optional per-session update cap
SESSION_HOURS = 9.5 if PROFILE == "overnight" else 0.0  # training time per stage
BENCHMARK_MICROBATCHES = 0              # 3 = time only 3 microbatches; use a fresh WORKDIR
VERBOSE = False                       # True: detailed per-worker diagnostics
DATA_SOURCE = None                    # optional local copy of published data
DATA_DIR = WORKDIR + "/data" if PROFILE == "overnight" else f"/kaggle/temp/latcomm-data-{PROFILE}"
PACKET_CACHE = WORKDIR + "/packets"    # full cache can require tens of GB
SKIP_SETUP = False                    # True only if this session's environment is installed
BOOTSTRAP_FILE = Path("/kaggle/working/latcomm_bootstrap.py")

os.environ["HF_HOME"] = "/kaggle/temp/latcomm-hf"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass


In [ ]:
# Bootstrap only; training code will be loaded from your fork.
BOOTSTRAP_FILE.write_text('"""Clone the user\'s LatComm fork, install the environment, run kaggle/run.py.\n\nThis entry point uses only Python\'s standard library and git. It can be copied\ninto a Kaggle cell or used by the accompanying lightweight notebook.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nREPO_URL = "https://github.com/SalavatFayzullin/LatComm.git"\nREQUIRED = ("kaggle/run.py", "kaggle/runtime.py", "kaggle/data_utils.py", "kaggle/progress.py", "latcomm/model.py")\n\n\ndef call(command, **kwargs):\n    print("+", " ".join(map(str, command)), flush=True)\n    return subprocess.run(list(map(str, command)), check=True, **kwargs)\n\n\ndef git(repo: Path, *arguments: str) -> str:\n    return subprocess.check_output(["git", "-C", str(repo), *arguments], text=True).strip()\n\n\ndef canonical_url(value: str) -> str:\n    return value.rstrip("/").removesuffix(".git")\n\n\ndef pinned_commit(args) -> str | None:\n    """A resumed experiment uses its saved SHA even when main has advanced."""\n    pins = []\n    for directory in (args.workdir, args.restore):\n        if directory is None:\n            continue\n        path = Path(directory) / "manifest.json"\n        if args.restore == directory and not path.is_file():\n            raise FileNotFoundError(f"Restore directory has no manifest.json: {directory}")\n        if not path.is_file():\n            continue\n        state = json.loads(path.read_text(encoding="utf-8"))\n        if canonical_url(state["source_url"]) != canonical_url(args.repo_url):\n            raise ValueError("Saved run belongs to another repository; choose another workdir.")\n        pins.append(state["source_commit"])\n    if args.commit:\n        pins.append(args.commit)\n    if len(set(pins)) > 1:\n        raise ValueError("Requested commit differs from the saved run; use its original commit or a new workdir.")\n    commit = pins[0] if pins else None\n    if commit is not None and not re.fullmatch(r"[0-9a-fA-F]{40}", commit):\n        raise ValueError("A pinned commit must be a full 40-character Git SHA.")\n    return commit\n\n\ndef checkout(args) -> tuple[Path, str]:\n    repo = args.repo_dir.resolve()\n    pin = pinned_commit(args)\n    fresh = False\n    if not args.ref or args.ref.startswith("-") or args.repo_url.startswith("-"):\n        raise ValueError("Invalid repository URL or ref.")\n    if not (repo / ".git").exists():\n        if repo.exists() and any(repo.iterdir()):\n            raise RuntimeError(f"Refusing to overwrite a nonempty directory: {repo}")\n        repo.parent.mkdir(parents=True, exist_ok=True)\n        call(["git", "clone", "--no-checkout", "--", args.repo_url, repo])\n        fresh = True\n    if canonical_url(git(repo, "remote", "get-url", "origin")) != canonical_url(args.repo_url):\n        raise RuntimeError(f"{repo} belongs to another repository; choose another --repo-dir.")\n    # Do not reset or overwrite local modifications in an existing Kaggle session.\n    if not fresh and git(repo, "status", "--porcelain", "--untracked-files=no"):\n        raise RuntimeError(f"Checkout has modified tracked files: {repo}. Keep it or choose another --repo-dir.")\n    if pin:\n        available = subprocess.run(["git", "-C", str(repo), "cat-file", "-e", f"{pin}^{{commit}}"],\n                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n        if available.returncode:\n            call(["git", "-C", repo, "fetch", "--no-tags", "origin", pin])\n        revision = pin\n    else:\n        call(["git", "-C", repo, "fetch", "--no-tags", "origin", args.ref])\n        revision = git(repo, "rev-parse", "FETCH_HEAD^{commit}")\n    call(["git", "-C", repo, "checkout", "--detach", revision])\n    commit = git(repo, "rev-parse", "HEAD")\n    missing = [name for name in REQUIRED if not (repo / name).is_file()]\n    if missing:\n        raise RuntimeError(\n            f"Fork revision {commit} is missing {\', \'.join(missing)}. "\n            "Commit and push the kaggle/ scripts to your fork, then select that branch or commit."\n        )\n    print(f"LatComm fork: {args.repo_url}\\nSource commit: {commit}", flush=True)\n    return repo, commit\n\n\ndef training_arguments(args):\n    command = [args.action, "--workdir", str(args.workdir), "--profile", args.profile,\n               "--gpus", str(args.gpus), "--microbatch", str(args.microbatch),\n               "--precision", args.precision, "--session-steps", str(args.session_steps),\n               "--session-hours", str(args.session_hours), "--checkpoint-every", str(args.checkpoint_every)]\n    if args.pair:\n        command += ["--pair", args.pair]\n    for name in ("restore", "data_source", "data_dir", "packet_cache", "data_revision"):\n        value = getattr(args, name)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    for name in ("audit_run", "training_repo", "audit_output"):\n        value = getattr(args, name, None)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    if getattr(args, "compare_stage1", False):\n        command += ["--compare-stage1"]\n    if args.benchmark_microbatches:\n        command += ["--benchmark-microbatches", str(args.benchmark_microbatches)]\n    if getattr(args, "verbose", False):\n        command += ["--verbose"]\n    return command\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-url", default=REPO_URL)\n    parser.add_argument("--ref", default="main", help="Branch/tag used for a new experiment.")\n    parser.add_argument("--commit", help="Optional full SHA; saved runs automatically pin their original SHA.")\n    parser.add_argument("--repo-dir", type=Path)\n    parser.add_argument("--workdir", type=Path)\n    parser.add_argument("--profile", choices=("kaggle", "overnight", "full", "smoke"), default="kaggle")\n    parser.add_argument("--pair", help="Omit to use the profile\'s model pair")\n    parser.add_argument("--action", choices=("setup", "prepare", "stage1", "encode", "stage2", "evaluate", "all", "audit"), default="all")\n    parser.add_argument("--gpus", type=int, default=0)\n    parser.add_argument("--microbatch", type=int, default=1)\n    parser.add_argument("--precision", choices=("auto", "fp16", "bf16"), default="auto")\n    parser.add_argument("--session-steps", type=int, default=0)\n    parser.add_argument("--session-hours", type=float, default=0)\n    parser.add_argument("--checkpoint-every", type=int, default=5)\n    parser.add_argument("--benchmark-microbatches", type=int, default=0)\n    parser.add_argument("--restore", type=Path)\n    parser.add_argument("--data-source", type=Path)\n    parser.add_argument("--data-dir", type=Path)\n    parser.add_argument("--packet-cache", type=Path)\n    parser.add_argument("--data-revision")\n    parser.add_argument("--skip-setup", action="store_true", help="Reuse an environment already installed in this session.")\n    parser.add_argument("--verbose", action="store_true", help="Show detailed per-worker training logs")\n    parser.add_argument("--audit-run", type=Path, help="Completed training workdir to evaluate without training")\n    parser.add_argument("--training-repo", type=Path, help="Optional unchanged source checkout of the completed run")\n    parser.add_argument("--audit-output", type=Path, help="New directory for control evaluation results")\n    parser.add_argument("--compare-stage1", action="store_true",\n                        help="Compare saved Stage 1 and Stage 2 bridges on identical MCQ controls")\n    args = parser.parse_args(argv)\n    default_workdir = "/kaggle/working/latcomm-controls" if args.action == "audit" else f"/kaggle/working/latcomm-{args.profile}"\n    args.workdir = (args.workdir or Path(default_workdir)).resolve()\n    default_repo = "/kaggle/working/LatComm-controls" if args.action == "audit" else "/kaggle/working/LatComm"\n    args.repo_dir = (args.repo_dir or Path(default_repo)).resolve()\n    if args.restore:\n        args.restore = args.restore.resolve()\n    if args.gpus < 0 or args.microbatch < 1 or args.checkpoint_every < 1 or args.session_steps < 0 or args.session_hours < 0 or args.benchmark_microbatches < 0:\n        parser.error("GPU count and session limits must be nonnegative; microbatch/checkpoint interval must be positive.")\n    if args.action == "audit" and (args.audit_run is None or args.restore is not None):\n        parser.error("audit requires --audit-run; use the completed run directly instead of --restore")\n    if args.compare_stage1 and args.action != "audit":\n        parser.error("--compare-stage1 requires --action audit")\n    if args.action == "audit":\n        if args.workdir == args.audit_run.resolve() or args.audit_run.resolve() in args.workdir.parents:\n            parser.error("Choose a separate --workdir for the control results")\n        if args.training_repo and args.repo_dir == args.training_repo.resolve():\n            parser.error("Choose a separate --repo-dir for the audit tools; preserve the training checkout")\n    repo, commit = checkout(args)\n    if args.action == "audit" and not (repo / "kaggle/audit.py").is_file():\n        raise RuntimeError(f"Fork revision {commit} has no control evaluator; commit and push kaggle/audit.py first")\n    script = repo / "kaggle/run.py"\n    env = dict(os.environ)\n    env["LATCOMM_SOURCE_COMMIT"] = commit\n    env["PYTHONPATH"] = os.pathsep.join([str(repo), str(repo / "kaggle"), env.get("PYTHONPATH", "")])\n    if not args.skip_setup or args.action == "setup":\n        call([sys.executable, script, "setup", "--workdir", args.workdir], cwd=repo, env=env)\n    if args.action == "setup":\n        return 0\n    command = [sys.executable, str(script), *training_arguments(args)]\n    print("+", " ".join(command), flush=True)\n    result = subprocess.run(command, cwd=repo, env=env)\n    if result.returncode == 75:\n        print("Session state saved. Resume with the same workdir or --restore in a later session.", flush=True)\n    return result.returncode\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding='utf-8')


## Клонирование, установка и запуск

Следующая ячейка клонирует форк, выводит фактический SHA, выполняет
`python kaggle/run.py setup`, затем выбранный `ACTION` из того же checkout.
Повторное клонирование репозитория авторов не выполняется.
Установка и обучение идут в отдельных Python-процессах.

Установщик использует PyTorch **2.11.0** и индекс cu126; Transformers — **5.15.1**.
Совместимая установленная сборка PyTorch может быть сохранена pip (например, cu128 на T4).
На этих GPU LLM работают в FP16, обучаемый интерфейс — в FP32, пакеты хранятся в BF16.
L2-нормализация query/key резервной реализации Qwen3.5 считается в FP32,
чтобы избежать NaN в её обратном проходе для малых FP16-векторов.
Это сохраняет алгоритм и отличается по вычислительной точности от H100 авторов.


In [ ]:
command = [sys.executable, str(BOOTSTRAP_FILE),
           "--repo-url", REPO_URL, "--ref", REPO_REF, "--repo-dir", REPO_DIR,
           "--workdir", WORKDIR, "--profile", PROFILE,
           "--action", ACTION, "--gpus", str(NUM_GPUS),
           "--session-steps", str(SESSION_STEPS), "--session-hours", str(SESSION_HOURS),
           "--benchmark-microbatches", str(BENCHMARK_MICROBATCHES),
           "--data-dir", DATA_DIR, "--packet-cache", PACKET_CACHE]
if SOURCE_COMMIT:
    command += ["--commit", SOURCE_COMMIT]
if PAIR:
    command += ["--pair", PAIR]
if RESTORE:
    command += ["--restore", RESTORE]
if DATA_SOURCE:
    command += ["--data-source", DATA_SOURCE]
if SKIP_SETUP:
    command += ["--skip-setup"]
if VERBOSE:
    command += ["--verbose"]
result = subprocess.run(command)
if result.returncode == 75:
    print("Сессия завершена штатно; сохраните Output и продолжите с RESTORE.")
elif result.returncode:
    raise RuntimeError(f"LatComm failed, exit code {result.returncode}. See logs above.")
elif PROFILE == "overnight" and ACTION == "all" and (Path(WORKDIR) / "results/metrics.json").is_file():
    controls_dir = Path(WORKDIR + "-controls")
    summary = controls_dir / "audit-results/audit_summary.json"
    if not summary.is_file():
        audit = subprocess.run([sys.executable, str(Path(REPO_DIR) / "kaggle/run.py"), "audit",
                                "--audit-run", WORKDIR, "--training-repo", REPO_DIR,
                                "--workdir", str(controls_dir), "--data-dir", DATA_DIR,
                                "--packet-cache", PACKET_CACHE, "--gpus", str(NUM_GPUS)])
        if audit.returncode:
            print("Контрольная оценка завершилась с ошибкой. Веса и основной тест сохранены в", WORKDIR)
    if summary.is_file():
        print("Сравнение контрольных режимов:", summary)


## Продолжение и результаты

Код 75 означает штатную паузу: сохранены optimizer, scheduler, scaler, RNG и позиция в данных.
Для следующей сессии сохраните Output, подключите его через Input и укажите `RESTORE`
на прежний `WORKDIR`, содержащий `manifest.json` и `outputs`.
Число GPU, микробатч, профиль и пару моделей при точном продолжении сохраняйте.

`/kaggle/temp` не сохраняется; JSONL и модели можно скачать повторно с закреплённых ревизий.
Кэш пакетов сохраняется отдельно. Его полный объём может превышать лимит сохраняемого output Kaggle.
Готовый совместимый кэш можно подключить из `/kaggle/input/...` через `PACKET_CACHE`.
Полный эксперимент требует многих сессий. Уменьшенный профиль выполняет 208 обновлений;
фактическое время и достаточность памяти нужно оценивать по текущему запуску.


In [ ]:
import json
benchmark = Path(WORKDIR) / "outputs/stage1/benchmark.json"
metrics = Path(WORKDIR) / "results/metrics.json"
if BENCHMARK_MICROBATCHES and benchmark.exists():
    print(json.dumps(json.loads(benchmark.read_text()), indent=2, ensure_ascii=False))
elif metrics.exists():
    print(json.dumps(json.loads(metrics.read_text()), indent=2, ensure_ascii=False))
else:
    print("Оценка ещё не завершена. Checkpoints:", str(Path(WORKDIR) / "outputs"))


## Контрольные оценки уже обученного интерфейса

Эта проверка использует готовый запуск. Повторного обучения нет.
Для текущей сессии оставьте `TRAINED_RUN`, `TRAINING_REPO` и `PREPARED_DATA`
как указано ниже. Измените `RUN_CONTROLS` на `True` и выполните **только следующую ячейку**.
Она самостоятельна: ячейки обучения повторно запускать не требуется.

На тех же тестовых вопросах будут сравнены:
обученный интерфейс с правильным пакетом; пакет другого контекста;
получатель без интерфейса и контекста; получатель с полным текстом контекста.
Во всех вариантах сохраняются вопросы, варианты и правило выбора ответа.
Подробные примеры CLOTH также сохраняются для проверки формата и ошибок.

Новый код проверок загружается в отдельный checkout. Исходный код обучения берётся
из SHA сохранённого запуска; модель, пакеты и прежние результаты не перезаписываются.
Результаты появятся в `CONTROLS_WORKDIR/audit-results`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

RUN_CONTROLS = False
COMPARE_STAGE1 = False
TRAINED_RUN = "/kaggle/working/latcomm-kaggle"
TRAINING_REPO = "/kaggle/working/LatComm"
PREPARED_DATA = "/kaggle/temp/latcomm-data-kaggle"
CONTROLS_WORKDIR = "/kaggle/working/latcomm-controls"
CONTROLS_REPO = "/kaggle/working/LatComm-controls"
CONTROLS_REF = "main"
CONTROLS_GPUS = 0                    # all visible GPUs
CONTROLS_SKIP_SETUP = True           # current session already has the training environment

if RUN_CONTROLS:
    os.environ.setdefault("HF_HOME", "/kaggle/temp/latcomm-hf")
    controls_bootstrap = Path("/kaggle/working/latcomm_controls_bootstrap.py")
    controls_bootstrap.write_text('"""Clone the user\'s LatComm fork, install the environment, run kaggle/run.py.\n\nThis entry point uses only Python\'s standard library and git. It can be copied\ninto a Kaggle cell or used by the accompanying lightweight notebook.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nREPO_URL = "https://github.com/SalavatFayzullin/LatComm.git"\nREQUIRED = ("kaggle/run.py", "kaggle/runtime.py", "kaggle/data_utils.py", "kaggle/progress.py", "latcomm/model.py")\n\n\ndef call(command, **kwargs):\n    print("+", " ".join(map(str, command)), flush=True)\n    return subprocess.run(list(map(str, command)), check=True, **kwargs)\n\n\ndef git(repo: Path, *arguments: str) -> str:\n    return subprocess.check_output(["git", "-C", str(repo), *arguments], text=True).strip()\n\n\ndef canonical_url(value: str) -> str:\n    return value.rstrip("/").removesuffix(".git")\n\n\ndef pinned_commit(args) -> str | None:\n    """A resumed experiment uses its saved SHA even when main has advanced."""\n    pins = []\n    for directory in (args.workdir, args.restore):\n        if directory is None:\n            continue\n        path = Path(directory) / "manifest.json"\n        if args.restore == directory and not path.is_file():\n            raise FileNotFoundError(f"Restore directory has no manifest.json: {directory}")\n        if not path.is_file():\n            continue\n        state = json.loads(path.read_text(encoding="utf-8"))\n        if canonical_url(state["source_url"]) != canonical_url(args.repo_url):\n            raise ValueError("Saved run belongs to another repository; choose another workdir.")\n        pins.append(state["source_commit"])\n    if args.commit:\n        pins.append(args.commit)\n    if len(set(pins)) > 1:\n        raise ValueError("Requested commit differs from the saved run; use its original commit or a new workdir.")\n    commit = pins[0] if pins else None\n    if commit is not None and not re.fullmatch(r"[0-9a-fA-F]{40}", commit):\n        raise ValueError("A pinned commit must be a full 40-character Git SHA.")\n    return commit\n\n\ndef checkout(args) -> tuple[Path, str]:\n    repo = args.repo_dir.resolve()\n    pin = pinned_commit(args)\n    fresh = False\n    if not args.ref or args.ref.startswith("-") or args.repo_url.startswith("-"):\n        raise ValueError("Invalid repository URL or ref.")\n    if not (repo / ".git").exists():\n        if repo.exists() and any(repo.iterdir()):\n            raise RuntimeError(f"Refusing to overwrite a nonempty directory: {repo}")\n        repo.parent.mkdir(parents=True, exist_ok=True)\n        call(["git", "clone", "--no-checkout", "--", args.repo_url, repo])\n        fresh = True\n    if canonical_url(git(repo, "remote", "get-url", "origin")) != canonical_url(args.repo_url):\n        raise RuntimeError(f"{repo} belongs to another repository; choose another --repo-dir.")\n    # Do not reset or overwrite local modifications in an existing Kaggle session.\n    if not fresh and git(repo, "status", "--porcelain", "--untracked-files=no"):\n        raise RuntimeError(f"Checkout has modified tracked files: {repo}. Keep it or choose another --repo-dir.")\n    if pin:\n        available = subprocess.run(["git", "-C", str(repo), "cat-file", "-e", f"{pin}^{{commit}}"],\n                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n        if available.returncode:\n            call(["git", "-C", repo, "fetch", "--no-tags", "origin", pin])\n        revision = pin\n    else:\n        call(["git", "-C", repo, "fetch", "--no-tags", "origin", args.ref])\n        revision = git(repo, "rev-parse", "FETCH_HEAD^{commit}")\n    call(["git", "-C", repo, "checkout", "--detach", revision])\n    commit = git(repo, "rev-parse", "HEAD")\n    missing = [name for name in REQUIRED if not (repo / name).is_file()]\n    if missing:\n        raise RuntimeError(\n            f"Fork revision {commit} is missing {\', \'.join(missing)}. "\n            "Commit and push the kaggle/ scripts to your fork, then select that branch or commit."\n        )\n    print(f"LatComm fork: {args.repo_url}\\nSource commit: {commit}", flush=True)\n    return repo, commit\n\n\ndef training_arguments(args):\n    command = [args.action, "--workdir", str(args.workdir), "--profile", args.profile,\n               "--gpus", str(args.gpus), "--microbatch", str(args.microbatch),\n               "--precision", args.precision, "--session-steps", str(args.session_steps),\n               "--session-hours", str(args.session_hours), "--checkpoint-every", str(args.checkpoint_every)]\n    if args.pair:\n        command += ["--pair", args.pair]\n    for name in ("restore", "data_source", "data_dir", "packet_cache", "data_revision"):\n        value = getattr(args, name)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    for name in ("audit_run", "training_repo", "audit_output"):\n        value = getattr(args, name, None)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    if getattr(args, "compare_stage1", False):\n        command += ["--compare-stage1"]\n    if args.benchmark_microbatches:\n        command += ["--benchmark-microbatches", str(args.benchmark_microbatches)]\n    if getattr(args, "verbose", False):\n        command += ["--verbose"]\n    return command\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-url", default=REPO_URL)\n    parser.add_argument("--ref", default="main", help="Branch/tag used for a new experiment.")\n    parser.add_argument("--commit", help="Optional full SHA; saved runs automatically pin their original SHA.")\n    parser.add_argument("--repo-dir", type=Path)\n    parser.add_argument("--workdir", type=Path)\n    parser.add_argument("--profile", choices=("kaggle", "overnight", "full", "smoke"), default="kaggle")\n    parser.add_argument("--pair", help="Omit to use the profile\'s model pair")\n    parser.add_argument("--action", choices=("setup", "prepare", "stage1", "encode", "stage2", "evaluate", "all", "audit"), default="all")\n    parser.add_argument("--gpus", type=int, default=0)\n    parser.add_argument("--microbatch", type=int, default=1)\n    parser.add_argument("--precision", choices=("auto", "fp16", "bf16"), default="auto")\n    parser.add_argument("--session-steps", type=int, default=0)\n    parser.add_argument("--session-hours", type=float, default=0)\n    parser.add_argument("--checkpoint-every", type=int, default=5)\n    parser.add_argument("--benchmark-microbatches", type=int, default=0)\n    parser.add_argument("--restore", type=Path)\n    parser.add_argument("--data-source", type=Path)\n    parser.add_argument("--data-dir", type=Path)\n    parser.add_argument("--packet-cache", type=Path)\n    parser.add_argument("--data-revision")\n    parser.add_argument("--skip-setup", action="store_true", help="Reuse an environment already installed in this session.")\n    parser.add_argument("--verbose", action="store_true", help="Show detailed per-worker training logs")\n    parser.add_argument("--audit-run", type=Path, help="Completed training workdir to evaluate without training")\n    parser.add_argument("--training-repo", type=Path, help="Optional unchanged source checkout of the completed run")\n    parser.add_argument("--audit-output", type=Path, help="New directory for control evaluation results")\n    parser.add_argument("--compare-stage1", action="store_true",\n                        help="Compare saved Stage 1 and Stage 2 bridges on identical MCQ controls")\n    args = parser.parse_args(argv)\n    default_workdir = "/kaggle/working/latcomm-controls" if args.action == "audit" else f"/kaggle/working/latcomm-{args.profile}"\n    args.workdir = (args.workdir or Path(default_workdir)).resolve()\n    default_repo = "/kaggle/working/LatComm-controls" if args.action == "audit" else "/kaggle/working/LatComm"\n    args.repo_dir = (args.repo_dir or Path(default_repo)).resolve()\n    if args.restore:\n        args.restore = args.restore.resolve()\n    if args.gpus < 0 or args.microbatch < 1 or args.checkpoint_every < 1 or args.session_steps < 0 or args.session_hours < 0 or args.benchmark_microbatches < 0:\n        parser.error("GPU count and session limits must be nonnegative; microbatch/checkpoint interval must be positive.")\n    if args.action == "audit" and (args.audit_run is None or args.restore is not None):\n        parser.error("audit requires --audit-run; use the completed run directly instead of --restore")\n    if args.compare_stage1 and args.action != "audit":\n        parser.error("--compare-stage1 requires --action audit")\n    if args.action == "audit":\n        if args.workdir == args.audit_run.resolve() or args.audit_run.resolve() in args.workdir.parents:\n            parser.error("Choose a separate --workdir for the control results")\n        if args.training_repo and args.repo_dir == args.training_repo.resolve():\n            parser.error("Choose a separate --repo-dir for the audit tools; preserve the training checkout")\n    repo, commit = checkout(args)\n    if args.action == "audit" and not (repo / "kaggle/audit.py").is_file():\n        raise RuntimeError(f"Fork revision {commit} has no control evaluator; commit and push kaggle/audit.py first")\n    script = repo / "kaggle/run.py"\n    env = dict(os.environ)\n    env["LATCOMM_SOURCE_COMMIT"] = commit\n    env["PYTHONPATH"] = os.pathsep.join([str(repo), str(repo / "kaggle"), env.get("PYTHONPATH", "")])\n    if not args.skip_setup or args.action == "setup":\n        call([sys.executable, script, "setup", "--workdir", args.workdir], cwd=repo, env=env)\n    if args.action == "setup":\n        return 0\n    command = [sys.executable, str(script), *training_arguments(args)]\n    print("+", " ".join(command), flush=True)\n    result = subprocess.run(command, cwd=repo, env=env)\n    if result.returncode == 75:\n        print("Session state saved. Resume with the same workdir or --restore in a later session.", flush=True)\n    return result.returncode\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
    command = [sys.executable, str(controls_bootstrap),
               "--repo-url", "https://github.com/SalavatFayzullin/LatComm.git",
               "--ref", CONTROLS_REF, "--repo-dir", CONTROLS_REPO,
               "--action", "audit", "--workdir", CONTROLS_WORKDIR,
               "--audit-run", TRAINED_RUN, "--training-repo", TRAINING_REPO,
               "--data-dir", PREPARED_DATA, "--packet-cache", str(Path(TRAINED_RUN) / "packets"),
               "--gpus", str(CONTROLS_GPUS)]
    if CONTROLS_SKIP_SETUP:
        command += ["--skip-setup"]
    if COMPARE_STAGE1:
        command += ["--compare-stage1"]
    subprocess.run(command, check=True)
    summary_file = Path(CONTROLS_WORKDIR) / (
        "stage-comparison-results/stage_comparison.json" if COMPARE_STAGE1
        else "audit-results/audit_summary.json")
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    names = {"actual_packet": "Правильный пакет", "shuffled_packet": "Чужой пакет",
             "query_only": "Без контекста", "raw_text": "Полный текст"}
    print(f"Сравнение на {summary['examples']} одинаковых вопросах")
    if COMPARE_STAGE1:
        stages = summary["stages"]
        print(f"{'Режим':20} {'Stage 1 acc':>12} {'Stage 2 acc':>12} {'Изменение':>11} {'Loss 1':>10} {'Loss 2':>10}")
        for mode in names:
            first = stages["stage1"]["modes"][mode]
            final = stages["stage2"]["modes"][mode]
            delta = 100 * (final["accuracy"] - first["accuracy"])
            print(f"{names[mode]:20} {first['accuracy']:>11.2%} {final['accuracy']:>11.2%} {delta:>+8.2f} п.п. {first['loss']:>10.4f} {final['loss']:>10.4f}")
        print("Воспроизведение прежнего Stage 2:", stages["stage2"].get("reproduction_check", {}).get("status", "не проверено"))
        print("\nПреимущество правильного пакета над чужим, п.п.:")
        print(f"{'Задача':15} {'Stage 1':>12} {'Stage 2':>12}")
        for task in sorted(stages["stage2"]["chance_accuracy"]["per_task"]):
            gains = [100 * (stages[stage]["modes"]["actual_packet"]["accuracy/" + task]
                            - stages[stage]["modes"]["shuffled_packet"]["accuracy/" + task])
                     for stage in ("stage1", "stage2")]
            print(f"{task:15} {gains[0]:>+12.2f} {gains[1]:>+12.2f}")
        print("\nИтог сравнения:", summary_file)
        print("Подробные ответы и примеры CLOTH:", summary_file.parent / "stage1", "и", summary_file.parent / "stage2")
    else:
        print(f"{'Режим':20} {'Верно':>8} {'Accuracy':>10} {'Loss':>10}")
        for mode, metrics in summary["modes"].items():
            correct = round(metrics["accuracy"] * summary["examples"])
            print(f"{names[mode]:20} {correct:>8} {metrics['accuracy']:>9.2%} {metrics['loss']:>10.4f}")
        print("Ожидаемая accuracy равномерного случайного выбора:", f"{summary['chance_accuracy']['overall']:.2%}")
        print("Воспроизведение прежнего результата:", summary.get("reproduction_check", {}).get("status", "не проверено"))
        print("\nТочность по задачам:")
        modes = list(summary["modes"])
        print(f"{'Задача':15}" + "".join(f"{names[mode]:>19}" for mode in modes))
        for task in sorted(summary["chance_accuracy"]["per_task"]):
            print(f"{task:15}" + "".join(f"{summary['modes'][mode]['accuracy/' + task]:>18.1%}" for mode in modes))
        print("\nПодробные результаты:", summary_file.parent)
        print("Примеры CLOTH:", summary_file.parent / "cloth_examples.jsonl")
else:
    print("Для оценки готового интерфейса задайте RUN_CONTROLS = True.")


## Сравнение сохранённых Stage 1 и Stage 2

Для локализации слабой зависимости от пакета эта самостоятельная ячейка проверяет
интерфейс после Stage 1 и финальный интерфейс после Stage 2 на одних тестовых вопросах.
Задайте `RUN_STAGE_COMPARISON = True` и выполните **только следующую ячейку**.
Повторного обучения и установки среды нет; используются сохранённые checkpoints и пакеты.

Для обоих интерфейсов выполняются те же четыре режима. В таблице будут точность,
loss и преимущество правильного пакета над чужим по задачам. Общий протокол:
`CONTROLS_WORKDIR/stage-comparison-results/stage_comparison.json`.
Полные ответы и примеры CLOTH сохраняются отдельно в подпапках `stage1` и `stage2`.
Прежняя папка `audit-results` сохраняется. Проверка Stage 1 на MCQ здесь диагностическая:
этот интерфейс обучался реконструкции текста, а не выбору буквы ответа.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

RUN_STAGE_COMPARISON = False
COMPARE_STAGE1 = True
TRAINED_RUN = "/kaggle/working/latcomm-kaggle"
TRAINING_REPO = "/kaggle/working/LatComm"
PREPARED_DATA = "/kaggle/temp/latcomm-data-kaggle"
CONTROLS_WORKDIR = "/kaggle/working/latcomm-controls"
CONTROLS_REPO = "/kaggle/working/LatComm-controls"
CONTROLS_REF = "main"
CONTROLS_GPUS = 0                    # all visible GPUs
CONTROLS_SKIP_SETUP = True           # current session already has the training environment

if RUN_STAGE_COMPARISON:
    os.environ.setdefault("HF_HOME", "/kaggle/temp/latcomm-hf")
    controls_bootstrap = Path("/kaggle/working/latcomm_controls_bootstrap.py")
    controls_bootstrap.write_text('"""Clone the user\'s LatComm fork, install the environment, run kaggle/run.py.\n\nThis entry point uses only Python\'s standard library and git. It can be copied\ninto a Kaggle cell or used by the accompanying lightweight notebook.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nREPO_URL = "https://github.com/SalavatFayzullin/LatComm.git"\nREQUIRED = ("kaggle/run.py", "kaggle/runtime.py", "kaggle/data_utils.py", "kaggle/progress.py", "latcomm/model.py")\n\n\ndef call(command, **kwargs):\n    print("+", " ".join(map(str, command)), flush=True)\n    return subprocess.run(list(map(str, command)), check=True, **kwargs)\n\n\ndef git(repo: Path, *arguments: str) -> str:\n    return subprocess.check_output(["git", "-C", str(repo), *arguments], text=True).strip()\n\n\ndef canonical_url(value: str) -> str:\n    return value.rstrip("/").removesuffix(".git")\n\n\ndef pinned_commit(args) -> str | None:\n    """A resumed experiment uses its saved SHA even when main has advanced."""\n    pins = []\n    for directory in (args.workdir, args.restore):\n        if directory is None:\n            continue\n        path = Path(directory) / "manifest.json"\n        if args.restore == directory and not path.is_file():\n            raise FileNotFoundError(f"Restore directory has no manifest.json: {directory}")\n        if not path.is_file():\n            continue\n        state = json.loads(path.read_text(encoding="utf-8"))\n        if canonical_url(state["source_url"]) != canonical_url(args.repo_url):\n            raise ValueError("Saved run belongs to another repository; choose another workdir.")\n        pins.append(state["source_commit"])\n    if args.commit:\n        pins.append(args.commit)\n    if len(set(pins)) > 1:\n        raise ValueError("Requested commit differs from the saved run; use its original commit or a new workdir.")\n    commit = pins[0] if pins else None\n    if commit is not None and not re.fullmatch(r"[0-9a-fA-F]{40}", commit):\n        raise ValueError("A pinned commit must be a full 40-character Git SHA.")\n    return commit\n\n\ndef checkout(args) -> tuple[Path, str]:\n    repo = args.repo_dir.resolve()\n    pin = pinned_commit(args)\n    fresh = False\n    if not args.ref or args.ref.startswith("-") or args.repo_url.startswith("-"):\n        raise ValueError("Invalid repository URL or ref.")\n    if not (repo / ".git").exists():\n        if repo.exists() and any(repo.iterdir()):\n            raise RuntimeError(f"Refusing to overwrite a nonempty directory: {repo}")\n        repo.parent.mkdir(parents=True, exist_ok=True)\n        call(["git", "clone", "--no-checkout", "--", args.repo_url, repo])\n        fresh = True\n    if canonical_url(git(repo, "remote", "get-url", "origin")) != canonical_url(args.repo_url):\n        raise RuntimeError(f"{repo} belongs to another repository; choose another --repo-dir.")\n    # Do not reset or overwrite local modifications in an existing Kaggle session.\n    if not fresh and git(repo, "status", "--porcelain", "--untracked-files=no"):\n        raise RuntimeError(f"Checkout has modified tracked files: {repo}. Keep it or choose another --repo-dir.")\n    if pin:\n        available = subprocess.run(["git", "-C", str(repo), "cat-file", "-e", f"{pin}^{{commit}}"],\n                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n        if available.returncode:\n            call(["git", "-C", repo, "fetch", "--no-tags", "origin", pin])\n        revision = pin\n    else:\n        call(["git", "-C", repo, "fetch", "--no-tags", "origin", args.ref])\n        revision = git(repo, "rev-parse", "FETCH_HEAD^{commit}")\n    call(["git", "-C", repo, "checkout", "--detach", revision])\n    commit = git(repo, "rev-parse", "HEAD")\n    missing = [name for name in REQUIRED if not (repo / name).is_file()]\n    if missing:\n        raise RuntimeError(\n            f"Fork revision {commit} is missing {\', \'.join(missing)}. "\n            "Commit and push the kaggle/ scripts to your fork, then select that branch or commit."\n        )\n    print(f"LatComm fork: {args.repo_url}\\nSource commit: {commit}", flush=True)\n    return repo, commit\n\n\ndef training_arguments(args):\n    command = [args.action, "--workdir", str(args.workdir), "--profile", args.profile,\n               "--gpus", str(args.gpus), "--microbatch", str(args.microbatch),\n               "--precision", args.precision, "--session-steps", str(args.session_steps),\n               "--session-hours", str(args.session_hours), "--checkpoint-every", str(args.checkpoint_every)]\n    if args.pair:\n        command += ["--pair", args.pair]\n    for name in ("restore", "data_source", "data_dir", "packet_cache", "data_revision"):\n        value = getattr(args, name)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    for name in ("audit_run", "training_repo", "audit_output"):\n        value = getattr(args, name, None)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    if getattr(args, "compare_stage1", False):\n        command += ["--compare-stage1"]\n    if args.benchmark_microbatches:\n        command += ["--benchmark-microbatches", str(args.benchmark_microbatches)]\n    if getattr(args, "verbose", False):\n        command += ["--verbose"]\n    return command\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-url", default=REPO_URL)\n    parser.add_argument("--ref", default="main", help="Branch/tag used for a new experiment.")\n    parser.add_argument("--commit", help="Optional full SHA; saved runs automatically pin their original SHA.")\n    parser.add_argument("--repo-dir", type=Path)\n    parser.add_argument("--workdir", type=Path)\n    parser.add_argument("--profile", choices=("kaggle", "overnight", "full", "smoke"), default="kaggle")\n    parser.add_argument("--pair", help="Omit to use the profile\'s model pair")\n    parser.add_argument("--action", choices=("setup", "prepare", "stage1", "encode", "stage2", "evaluate", "all", "audit"), default="all")\n    parser.add_argument("--gpus", type=int, default=0)\n    parser.add_argument("--microbatch", type=int, default=1)\n    parser.add_argument("--precision", choices=("auto", "fp16", "bf16"), default="auto")\n    parser.add_argument("--session-steps", type=int, default=0)\n    parser.add_argument("--session-hours", type=float, default=0)\n    parser.add_argument("--checkpoint-every", type=int, default=5)\n    parser.add_argument("--benchmark-microbatches", type=int, default=0)\n    parser.add_argument("--restore", type=Path)\n    parser.add_argument("--data-source", type=Path)\n    parser.add_argument("--data-dir", type=Path)\n    parser.add_argument("--packet-cache", type=Path)\n    parser.add_argument("--data-revision")\n    parser.add_argument("--skip-setup", action="store_true", help="Reuse an environment already installed in this session.")\n    parser.add_argument("--verbose", action="store_true", help="Show detailed per-worker training logs")\n    parser.add_argument("--audit-run", type=Path, help="Completed training workdir to evaluate without training")\n    parser.add_argument("--training-repo", type=Path, help="Optional unchanged source checkout of the completed run")\n    parser.add_argument("--audit-output", type=Path, help="New directory for control evaluation results")\n    parser.add_argument("--compare-stage1", action="store_true",\n                        help="Compare saved Stage 1 and Stage 2 bridges on identical MCQ controls")\n    args = parser.parse_args(argv)\n    default_workdir = "/kaggle/working/latcomm-controls" if args.action == "audit" else f"/kaggle/working/latcomm-{args.profile}"\n    args.workdir = (args.workdir or Path(default_workdir)).resolve()\n    default_repo = "/kaggle/working/LatComm-controls" if args.action == "audit" else "/kaggle/working/LatComm"\n    args.repo_dir = (args.repo_dir or Path(default_repo)).resolve()\n    if args.restore:\n        args.restore = args.restore.resolve()\n    if args.gpus < 0 or args.microbatch < 1 or args.checkpoint_every < 1 or args.session_steps < 0 or args.session_hours < 0 or args.benchmark_microbatches < 0:\n        parser.error("GPU count and session limits must be nonnegative; microbatch/checkpoint interval must be positive.")\n    if args.action == "audit" and (args.audit_run is None or args.restore is not None):\n        parser.error("audit requires --audit-run; use the completed run directly instead of --restore")\n    if args.compare_stage1 and args.action != "audit":\n        parser.error("--compare-stage1 requires --action audit")\n    if args.action == "audit":\n        if args.workdir == args.audit_run.resolve() or args.audit_run.resolve() in args.workdir.parents:\n            parser.error("Choose a separate --workdir for the control results")\n        if args.training_repo and args.repo_dir == args.training_repo.resolve():\n            parser.error("Choose a separate --repo-dir for the audit tools; preserve the training checkout")\n    repo, commit = checkout(args)\n    if args.action == "audit" and not (repo / "kaggle/audit.py").is_file():\n        raise RuntimeError(f"Fork revision {commit} has no control evaluator; commit and push kaggle/audit.py first")\n    script = repo / "kaggle/run.py"\n    env = dict(os.environ)\n    env["LATCOMM_SOURCE_COMMIT"] = commit\n    env["PYTHONPATH"] = os.pathsep.join([str(repo), str(repo / "kaggle"), env.get("PYTHONPATH", "")])\n    if not args.skip_setup or args.action == "setup":\n        call([sys.executable, script, "setup", "--workdir", args.workdir], cwd=repo, env=env)\n    if args.action == "setup":\n        return 0\n    command = [sys.executable, str(script), *training_arguments(args)]\n    print("+", " ".join(command), flush=True)\n    result = subprocess.run(command, cwd=repo, env=env)\n    if result.returncode == 75:\n        print("Session state saved. Resume with the same workdir or --restore in a later session.", flush=True)\n    return result.returncode\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
    command = [sys.executable, str(controls_bootstrap),
               "--repo-url", "https://github.com/SalavatFayzullin/LatComm.git",
               "--ref", CONTROLS_REF, "--repo-dir", CONTROLS_REPO,
               "--action", "audit", "--workdir", CONTROLS_WORKDIR,
               "--audit-run", TRAINED_RUN, "--training-repo", TRAINING_REPO,
               "--data-dir", PREPARED_DATA, "--packet-cache", str(Path(TRAINED_RUN) / "packets"),
               "--gpus", str(CONTROLS_GPUS)]
    if CONTROLS_SKIP_SETUP:
        command += ["--skip-setup"]
    if COMPARE_STAGE1:
        command += ["--compare-stage1"]
    subprocess.run(command, check=True)
    summary_file = Path(CONTROLS_WORKDIR) / (
        "stage-comparison-results/stage_comparison.json" if COMPARE_STAGE1
        else "audit-results/audit_summary.json")
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    names = {"actual_packet": "Правильный пакет", "shuffled_packet": "Чужой пакет",
             "query_only": "Без контекста", "raw_text": "Полный текст"}
    print(f"Сравнение на {summary['examples']} одинаковых вопросах")
    if COMPARE_STAGE1:
        stages = summary["stages"]
        print(f"{'Режим':20} {'Stage 1 acc':>12} {'Stage 2 acc':>12} {'Изменение':>11} {'Loss 1':>10} {'Loss 2':>10}")
        for mode in names:
            first = stages["stage1"]["modes"][mode]
            final = stages["stage2"]["modes"][mode]
            delta = 100 * (final["accuracy"] - first["accuracy"])
            print(f"{names[mode]:20} {first['accuracy']:>11.2%} {final['accuracy']:>11.2%} {delta:>+8.2f} п.п. {first['loss']:>10.4f} {final['loss']:>10.4f}")
        print("Воспроизведение прежнего Stage 2:", stages["stage2"].get("reproduction_check", {}).get("status", "не проверено"))
        print("\nПреимущество правильного пакета над чужим, п.п.:")
        print(f"{'Задача':15} {'Stage 1':>12} {'Stage 2':>12}")
        for task in sorted(stages["stage2"]["chance_accuracy"]["per_task"]):
            gains = [100 * (stages[stage]["modes"]["actual_packet"]["accuracy/" + task]
                            - stages[stage]["modes"]["shuffled_packet"]["accuracy/" + task])
                     for stage in ("stage1", "stage2")]
            print(f"{task:15} {gains[0]:>+12.2f} {gains[1]:>+12.2f}")
        print("\nИтог сравнения:", summary_file)
        print("Подробные ответы и примеры CLOTH:", summary_file.parent / "stage1", "и", summary_file.parent / "stage2")
    else:
        print(f"{'Режим':20} {'Верно':>8} {'Accuracy':>10} {'Loss':>10}")
        for mode, metrics in summary["modes"].items():
            correct = round(metrics["accuracy"] * summary["examples"])
            print(f"{names[mode]:20} {correct:>8} {metrics['accuracy']:>9.2%} {metrics['loss']:>10.4f}")
        print("Ожидаемая accuracy равномерного случайного выбора:", f"{summary['chance_accuracy']['overall']:.2%}")
        print("Воспроизведение прежнего результата:", summary.get("reproduction_check", {}).get("status", "не проверено"))
        print("\nТочность по задачам:")
        modes = list(summary["modes"])
        print(f"{'Задача':15}" + "".join(f"{names[mode]:>19}" for mode in modes))
        for task in sorted(summary["chance_accuracy"]["per_task"]):
            print(f"{task:15}" + "".join(f"{summary['modes'][mode]['accuracy/' + task]:>18.1%}" for mode in modes))
        print("\nПодробные результаты:", summary_file.parent)
        print("Примеры CLOTH:", summary_file.parent / "cloth_examples.jsonl")
else:
    print("Для сравнения сохранённых Stage 1 и Stage 2 задайте RUN_STAGE_COMPARISON = True.")


Это запуск LatComm для Kaggle **из своего форка [SalavatFayzullin/LatComm](https://github.com/SalavatFayzullin/LatComm)**. По умолчанию выбран уменьшенный профиль `kaggle` с парой Qwen3.5-0.8B → Qwen3.5-0.8B. Все модули и конфигурации берутся из одного checkout. Файлы `latcomm/bridge.py` и `latcomm/model.py` не переписаны; в функции потерь добавлен режим пересчёта чанков для экономии памяти при сохранении CE и её градиентов.

Перед первым запуском закоммитьте и отправьте папку `kaggle/` в выбранную ветку форка. Например, из корня локального репозитория:

```bash
git add kaggle latcomm/evaluate.py tests/test_loss_memory.py README.md
git commit -m "Add reduced Kaggle experiment and memory-saving loss"
git push origin main
```

Импортируйте `kaggle/LatComm_Kaggle.ipynb` в Kaggle. Выберите **GPU T4 ×2** (предпочтительно) или **P100**, включите **Internet**, затем запускайте ячейки сверху вниз. Ноутбук содержит только bootstrap: он клонирует ваш форк в `/kaggle/working/LatComm`, запускает `python kaggle/run.py setup`, затем `python kaggle/run.py all` из того же репозитория. Установка и обучение выполняются в отдельных Python-процессах.

`REPO_REF = "main"` выбирает ветку для нового эксперимента. `SOURCE_COMMIT` позволяет указать конкретный полный SHA. Фактический коммит записывается в `manifest.json`; при повторном запуске или `RESTORE` bootstrap автоматически выбирает прежний SHA, даже если ветка main обновилась. Для эксперимента с обновлённым кодом выберите новый `WORKDIR`. Существующий checkout с изменёнными tracked-файлами автоматически не перезаписывается.

Данные уже опубликованы авторами: [huaweilin/LatComm](https://huggingface.co/datasets/huaweilin/LatComm). Загрузчик использует исходные сжатые JSONL, сохраняя разбиения и выбранные записи без переформатирования. Размеры исходного полного набора:

| Этап | Данные | Train | Validation | Test |
| --- | --- | ---: | ---: | ---: |
| Stage 1 | WikiText-103 + FineWeb-Edu, восстановление текста | 2 000 000 | 15 043 | 14 983 |
| Stage 2 | 12 наборов вопросов с выбором ответа | 282 441 | 2 036 | 55 890 |

На Stage 2 входят CLOTH, Cosmos QA, HellaSwag, LogiQA, MCScript2.0, MuTual, OpenBookQA, QuALITY, ReClor, SciQ, SocialIQA и SWAG. RACE, DREAM и QuAIL — дополнительные эксперименты переноса из статьи; данный запуск оценивает опубликованный основной тестовый набор из 12 задач.

По умолчанию `PROFILE = "kaggle"` использует следующие подмножества:

| Этап | Train | Проверка во время обучения | Test | Эпохи | Global batch | Обновления |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Stage 1 | 2048 текстов | 64 | 64 | 1 | 32 | 64 |
| Stage 2 | 1536 вопросов (128 на набор) | 48 (4 на набор) | 192 (16 на набор) | 3 | 32 | 144 |

Stage 1 выбирается равномерной потоковой выборкой из всего исходного split. Stage 2 выбирается отдельно для каждого из 12 наборов, чтобы большие наборы не вытеснили маленькие. Seed 42 фиксирован. В новом профиле `fast_dev` Stage 2 совпадает с выбранным `dev`. Параметры выборки записываются в `preparation.json`; сами тексты, варианты и ответы сохраняются. Это уменьшенный эксперимент с методом статьи; его метрики не воспроизводят её таблицы.

В настройках ноутбука:

- `PROFILE = "kaggle"` — уменьшенный эксперимент выше. На двух T4 микробатч 1 и накопление 16 дают global batch 32. Warmup масштабирован относительно общего числа обновлений: 2 шага для Stage 1 и 3 для Stage 2. Learning rate и виды scheduler сохраняются.
- `PROFILE = "overnight"` — 45 056 текстов Stage 1, одна эпоха, 1 408 обновлений и 42 шага warmup. Остальные выборки совпадают с `kaggle` при той же ревизии датасета. Stage 2, пара моделей и global batch 32 сохраняются. На двух T4 по темпу пилота обучение, кодирование и оценка займут суммарно около 15 часов, плюс подготовка окружения.
- `PROFILE = "full"` явно включает все данные статьи и batch 384. Эпохи: 1 для Stage 1, затем 3 для Stage 2. Основная пара: Qwen3.5-4B → Qwen3.5-0.8B. По наблюдаемому раннему темпу полного запуска один Stage 1 экстраполировался примерно в 29 суток; такой режим непрактичен для данного запуска на двух T4.
- `PROFILE = "smoke"` — явно уменьшенная проверка всей цепочки на небольшом числе примеров. Её результаты нельзя сравнивать с таблицами статьи. Ноутбук использует отдельный `WORKDIR` для каждого профиля. Загрузка исходного архива Stage 1 (около 2,8 GB) всё равно выполняется целиком; уменьшается число распакованных и используемых записей.
- `SESSION_STEPS = 0` и `SESSION_HOURS = 0` отключают дополнительные остановки скрипта. Для `overnight` в ноутбуке `SESSION_HOURS = 9.5`. Это время обучения отдельного этапа, без установки, подготовки данных и первоначальной оценки. Остановка происходит на границе обновления с сохранением состояния.
- `ACTION = "all"` последовательно выполняет Stage 1 → сохранение пакетов → Stage 2 → оценку. При достижении границы сессии цепочка останавливается. Следующий запуск продолжает незавершённый этап.
- `PAIR = None` автоматически выбирает пару для профиля: 0.8B → 0.8B для `kaggle` и `overnight`, 4B → 0.8B для `full` и `smoke`. Меньшая пара также исследована в статье. Пару можно задать явно; меняйте `WORKDIR` вместе с ней.

У [GPU-сессий Kaggle](https://www.kaggle.com/docs/notebooks) лимит 12 часов. Для `overnight` выберите T4 ×2, включите Internet и установите `PROFILE = "overnight"` в обновлённом ноутбуке. С `RESTORE = None` ноутбук запускает только Stage 1. Запустите **Save Version → Save & Run All**, чтобы Kaggle опубликовал Output после штатной паузы. В чистой сессии подключите этот Output как Input и укажите `RESTORE` на вложенный каталог `latcomm-overnight` с `manifest.json`. Оставьте T4 ×2 и те же настройки среды. Повторный **Save & Run All** продолжит с чекпоинта, построит пакеты, выполнит Stage 2 и оценку. Ориентир для второй части при прежнем темпе — 5–6 часов вычислений.

Для `overnight` подготовленные данные лежат в `WORKDIR/data` и переносятся вместе с состоянием при `RESTORE`. Веса, настройки, данные, финальные метрики и ответы остаются в Output. После успешного финального теста ноутбук запускает существующие контрольные режимы и сохраняет сводку в `/kaggle/working/latcomm-overnight-controls/audit-results/audit_summary.json`.

Архитектура и функция потерь сохраняются: пакет строится только из контекста, без вопроса; learned queries и cross-attention сжимают его в 32 слота ширины 256 на пару слоёв; gates зависят только от prompt positions; обе LLM заморожены; Stage 1 обучает весь интерфейс, Stage 2 фиксирует compressors и обучает readers/gates. Loss усредняется по всем ответным токенам глобального обновления. Тексты не обрезаются молча. Пакеты сохраняются в BF16 и дедуплицируются по контексту. В `kaggle` изменены модель encoder, объём выборки, batch и длина warmup; эти отличия отражены в конфигурации.

Численная конфигурация отличается от H100 в статье: T4/P100 используют FP16 для LLM, FP32 для интерфейса и AdamW, GradScaler для градиентов. Опубликованный код авторов также считает интерфейс в FP32. При overflow повторяется та же группа микробатчей с меньшим scale, а повторяющаяся ошибка завершает процесс. Потеря не должна становиться NaN молча. Для CE используется non-reentrant checkpointing по 32 позициям: большие промежуточные FP32-буферы не сохраняются одновременно для всего текста. Проверки сверяют значения CE и её градиенты с обычной реализацией. Квантизация и LoRA не используются.

В FP16 runtime переводит L2-нормализацию query/key резервной реализации Qwen3.5 в FP32 и возвращает результат в FP16. В Transformers 5.15.1 её промежуточные значения при backward для нулевых и малых FP16-векторов переполняются, хотя forward остаётся конечным. Это воспроизводится локально; снижение loss scale такой случай не устраняет. Формула и epsilon сохраняются. Исправление также проверяется с настоящими слоями Qwen3.5 малого размера, reader hooks, CE и gradient checkpointing; веса для этой проверки не скачиваются. BF16-режим не меняется. В подробном режиме выводится `FP16 stability: Qwen3.5 fallback Q/K L2 normalization computes in FP32`.

Если прежний запуск повторяет `next_step: 1` с предупреждениями `fp16_retry`, отправьте исправленные файлы в форк и остановите тот запуск. В том же ноутбуке задайте `WORKDIR = "/kaggle/working/latcomm-kaggle-fp16fix"`, `SOURCE_COMMIT = None`, `RESTORE = None` и выполните ячейки заново. `PACKET_CACHE` должен вычисляться из нового `WORKDIR`; `DATA_DIR` и `HF_HOME` можно сохранить. Новый каталог нужен, чтобы bootstrap взял исправленный код из main: существующий `manifest.json` закрепляет старый SHA. Старые checkpoints сохраняются в прежнем каталоге.

Корпус Stage 1 занимает около 2,8 GB в сжатом виде и для уменьшенной выборки тоже скачивается целиком, затем просматривается потоком; 2 млн записей в RAM не материализуются. Один стандартный пакет занимает 384 KiB. В уменьшенном профиле кэш максимум 1776 контекстов составляет около 650 MiB до дедупликации; в полном режиме это могут быть десятки GB. Runtime проверяет диск перед кодированием. Две T4 работают через data parallel: каждая хранит собственные копии моделей. Новая меньшая пара снижает объём весов на каждой GPU, но GPU-пиковая память и время здесь пока не измерены. Само сообщение Kernel died без traceback не устанавливает причину падения.

Для продолжения сохраните notebook outputs и подключите их к следующей сессии как Input. Укажите `RESTORE` на прежнюю папку `latcomm`, содержащую `manifest.json` и `outputs`. Runtime сохраняет bridge, optimizer, scheduler, scaler, RNG и позицию в данных на границе обновления. Он проверяет идентичность данных, моделей, кода и настроек; число GPU и микробатч при точном продолжении менять нельзя. Сохраняются два последних промежуточных checkpoints и финальный.

Каталоги `/kaggle/temp` не сохраняются в output. В ноутбуке туда направлены скачанные модели и JSONL, чтобы не заполнять постоянный output. Данные можно скачать заново с закреплённой ревизии. Пакеты по умолчанию направлены в `WORKDIR/packets`; для полного запуска потребуется достаточно диска и отдельное сохранение кэша. Уже готовый кэш можно подключить через `PACKET_CACHE = "/kaggle/input/.../packets"`; его не нужно копировать. Для продолжения незавершённого кодирования кэш должен быть доступен на запись.

В уже клонированном репозитории можно запускать команды напрямую из его корня:

```bash
python kaggle/run.py setup
python kaggle/run.py all --profile kaggle --gpus 2
```

Отдельный `kaggle/bootstrap.py` также умеет клонировать форк, устанавливать зависимости и запускать обучение: `python kaggle/bootstrap.py --profile kaggle --gpus 2`. `run.py` использует репозиторий, в котором находится сам, и ничего дополнительно не клонирует.

Используется один ноутбук `LatComm_Kaggle.ipynb`. Для обычного обучения оставьте `BENCHMARK_MICROBATCHES = 0`. При необходимости необязательный замер можно включить в том же ноутбуке, указав число микробатчей и новый `WORKDIR`; отдельный ноутбук не нужен.

По умолчанию `VERBOSE = False`: консоль показывает переходы между этапами, начало и завершение эпох, шаг относительно общего числа обновлений, loss/accuracy и приблизительный ETA текущего этапа обучения. Обновления прогресса появляются примерно раз в 30 секунд; первый завершённый шаг и конец каждой эпохи выводятся обязательно. Во время длинного шага виден счётчик микробатчей. Обычный прогресс выводит только ведущий процесс, без дублей от второй GPU. Из памяти в сводке остаётся только занятая RAM контейнера относительно лимита. HTTP-запросы, JSON для каждой фазы, подробная статистика VRAM и перечни уже установленных pip-пакетов убраны из обычного вывода. Предупреждения о численной нестабильности и ошибки сохраняются.

Создание пакетов также выводит прогресс раз в 30 секунд: считается общее число контекстов в опубликованных индексах обеих GPU. Ещё не сохранённые пакеты не включаются в счётчик. При длительной проверке показываются обработанные батчи ведущего процесса; итоговые метрики собираются по всем GPU. Отчёты появляются после завершения очередного микробатча/батча; отдельного фонового heartbeat внутри блокирующей GPU-операции нет. ETA обучения использует среднее время последних десяти обновлений и не включает будущие проверки, сохранения и другие этапы.

Полные метрики каждого обновления и проверки остаются в `outputs/stage1/metrics.jsonl` и `outputs/stage2/metrics.jsonl`. Финальные результаты по всем задачам — в `results/metrics.json`. Для подробных фаз, диагностики памяти и сообщений каждого процесса включите `VERBOSE = True` в том же ноутбуке либо передайте `--verbose` в `bootstrap.py`, `run.py` или `runtime.py`. Промежуточные checkpoints продолжают сохраняться с прежней частотой, их сообщения видны в подробном режиме; финальное сохранение и штатная пауза видны всегда.

Для контрольной оценки уже завершённого запуска используется тот же ноутбук. Сначала отправьте обновлённую папку `kaggle/` в форк. Затем скопируйте **весь** файл `kaggle/Run_LatComm_Controls.py` в **новую ячейку текущего Kaggle-ноутбука** и выполните её. В этой ячейке проверка уже включена. Переимпортировать ноутбук и выполнять ячейки обучения не требуется. В обновлённом `LatComm_Kaggle.ipynb` такая же самостоятельная ячейка есть после результатов; там для запуска нужно изменить `RUN_CONTROLS = False` на `True`.

Настройки для текущей сессии уже заполнены: `TRAINED_RUN = "/kaggle/working/latcomm-kaggle"`, `TRAINING_REPO = "/kaggle/working/LatComm"`, `PREPARED_DATA = "/kaggle/temp/latcomm-data-kaggle"`. `CONTROLS_SKIP_SETUP = True` сохраняет уже установленное окружение. Новый код проверок клонируется отдельно в `/kaggle/working/LatComm-controls`; результаты записываются в `/kaggle/working/latcomm-controls/audit-results`. Для повторной проверки выберите другой `CONTROLS_WORKDIR`: непустой каталог результатов не перезаписывается. Подготовленный `stage2/test.jsonl` и `preparation.json` должны сохраниться в `PREPARED_DATA`; новая выборка вместо прежних тестовых вопросов не создаётся.

Контроли оценивают ровно те же вопросы, без обучения, с batch 1 и исходным правилом выбора буквы:

| Режим | Что получает получатель |
| --- | --- |
| `actual_packet` | Правильный пакет и обученный интерфейс |
| `shuffled_packet` | Пакет другого контекста той же задачи и тот же интерфейс |
| `query_only` | Вопрос и варианты; весь интерфейс отключён |
| `raw_text` | Полный текст контекста перед тем же вопросом; весь интерфейс отключён |

Перестановка строится с seed 42 по уникальным контекстам отдельно для каждой задачи: собственный контекст не может попасть к своему вопросу. Если в какой-то задаче только один уникальный контекст, проверка завершается с объяснением. Слишком длинный полный текст не обрезается и вопросы не исключаются: проверка явно сообщает о превышении окна модели. Формат raw-text — `Context:\n{context}\n\n{original MCQ prompt}`; он записан в протоколе результатов. Равномерный случайный уровень вычисляется из реального числа вариантов каждого вопроса.

Модель и scoring импортируются из **точного коммита прежнего обучения**, а не из нового runtime. Если исходный checkout изменён, его прежний SHA клонируется в отдельную папку внутри `CONTROLS_WORKDIR`. Проверяются хеши исходного кода, версии библиотек, ревизии моделей, исходный тест, параметры компрессора и кэш. Папка обучения, её checkpoints, данные и прежние результаты читаются без перезаписи. Проверка не меняет правила точного resume. Она использует **финальную модель прежнего запуска**, а не восстанавливает отсутствующий checkpoint второй эпохи.

В ячейке выводится компактная таблица accuracy/loss четырёх режимов и таблица по задачам. `reproduction_check = match` означает, что правильный пакет воспроизвёл прежние метрики и predictions в заданной численной точности. `different` требует разбора указанных sample IDs, а `unavailable` означает отсутствие достаточных прежних результатов для такого сравнения. Файлы: `audit_summary.json`, `predictions.jsonl`, `metrics_<mode>.json`, `test_source_records.jsonl`, `input_lengths.json` и `cloth_examples.jsonl`. В CLOTH export сохранены весь исходный record, контекст, фактический prompt, правильная буква и ответы всех режимов. Отличия между режимами помогают локализовать проблему; сами по себе они не доказывают единственную причину ошибок.

Чтобы сравнить интерфейс **до и после Stage 2**, отправьте изменения в форк и скопируйте **весь** `kaggle/Run_LatComm_Stage_Comparison.py` в **новую ячейку того же работающего Kaggle-ноутбука**. Выполните только эту ячейку. В ней `RUN_STAGE_COMPARISON = True` уже установлен; в обновлённом основном ноутбуке соответствующая ячейка по умолчанию выключена. Настройки путей совпадают с контрольной оценкой выше. `CONTROLS_SKIP_SETUP = True` сохраняет текущее окружение. Перезапускать kernel, импортировать второй ноутбук или запускать обучение не требуется.

Проверка читает `outputs/stage1/checkpoints/final` и `outputs/stage2/checkpoints/final` прежнего запуска. Обоим интерфейсам задаются те же исходные тестовые вопросы и все четыре контрольных режима. Используется один и тот же кэш пакетов: параметры компрессоров должны совпадать между Stage 1, Stage 2 и кэшем, иначе проверка завершится с объяснением. Модель-получатель, tokenization, MCQ prompt, правило выбора буквы и перестановка пакетов одинаковы для обоих этапов. Обновлений весов нет. Stage 1 обучался реконструкции текста; его MCQ-результат показывает поведение до настройки на ответы-буквы и не заменяет оценку качества реконструкции.

Результаты сравнения записываются отдельно: `/kaggle/working/latcomm-controls/stage-comparison-results/stage_comparison.json`. Там находятся сводки обоих этапов и парные изменения ответов для каждого режима. В подпапках `stage1` и `stage2` сохраняются полные `audit_summary.json`, `predictions.jsonl`, `cloth_examples.jsonl` и остальные экспорты. Таблица в ячейке показывает accuracy/loss до и после Stage 2, а также преимущество правильного пакета над чужим для каждой задачи. `reproduction_check` проверяет Stage 2 относительно прежнего финального теста; для Stage 1 прежнего MCQ-теста не было. Папка прежних контролей `audit-results` сохраняется; повторный запуск сравнения требует другого `CONTROLS_WORKDIR`, поскольку непустые результаты не перезаписываются.

Если до Stage 2 правильный пакет помогает заметнее, чем после, это указывает на ухудшение использования контекста во время MCQ-настройки. Если разницы почти нет уже после Stage 1, полезно отдельно исследовать реконструкцию и передачу сигнала. Одно такое сравнение не устанавливает единственную причину: Stage 1 ещё не обучен отвечать буквами, а тест уменьшенного эксперимента мал. Эта проверка не восстанавливает отсутствующие веса шага 96 и не создаёт их повторным обучением.

В **будущих новых обучениях** дополнительно сохраняется bridge с наименьшим конечным validation loss: `outputs/<stage>/best.json` указывает на `checkpoints/best-step-XXXXXXXX`. При равном loss сохраняется прежний выбор. Stage 1 step 0 исключён, Stage 2 step 0 участвует. Сохраняется один лучший bridge отдельно от двух последних точных training-checkpoints и `final`; optimizer/RNG/эпохи и `latest.json` не меняются. Best bridge предназначен для отдельной оценки, точное продолжение требует полного training checkpoint. Переход Stage 1 → Stage 2 и стандартная финальная оценка по-прежнему используют `final`. Старый завершённый запуск не переобучается и не получает задним числом промежуточные веса.

Для диагностики используется отдельная папка `/kaggle/working/latcomm-benchmark`. Отправьте обновлённые скрипты в форк перед запуском. Существующий `WORKDIR` закрепляет старый SHA, поэтому обновление main само по себе его код не меняет. Строгая проверка resume отклоняет checkpoints от другой версии runtime; диагностика не меняет старые checkpoints. В той же сессии уже скачанные модели и исходные архивы могут быть повторно использованы из прежнего HF cache.

Оценка пишет `results/metrics.json` и `predictions.jsonl`: точность по каждой задаче, общую точность и среднее по наборам. Версии моделей и датасета фиксируются при первом запуске в `manifest.json`; повторное обучение с изменёнными данными требует отдельного `WORKDIR`.

Проверки в комплекте выполняются без скачивания LLM. Полное обучение на реальных GPU Kaggle в этой локальной среде не выполнялось.

Источники: [официальная реализация](https://github.com/huawei-lin/LatComm), [статья](https://arxiv.org/abs/2610.26840), [совместимость CUDA wheels с P100](https://dev-discuss.pytorch.org/t/notice-cuda-12-6-wheels-will-no-longer-be-published-from-pytorch-2-15-drops-maxwell-pascal-volta/3432).
